In [ ]:
import os
import sys
import numpy as np
import datetime
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
# ── Training & evaluation utilities ──────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def train(model, train_loader, criterion, optimizer, device):
    """One full epoch over train_loader. Returns mean batch loss."""
    model.train()   # enables dropout and batch-norm update
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()           # clear gradients from the previous batch
        out  = model(X)
        loss = criterion(out, y)        # cross-entropy between logits and true labels
        loss.backward()                 # backprop: compute gradients for all parameters
        optimizer.step()               # update weights using the computed gradients
        total_loss += loss.item()
    return total_loss / len(train_loader)   # mean loss across all batches


def evaluate(model, loader, device):
    """Returns top-1 accuracy over loader. No gradient computation."""
    model.eval()    # disables dropout, freezes batch-norm stats
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)   # predicted class = index of highest logit
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return correct / total


def evaluateFinal(model, test_loader, device, plot=True):
    """Full evaluation with optional confusion matrix — use for post-hoc analysis."""
    model.eval()
    correct, total = 0, 0
    actual    = torch.tensor([], dtype=torch.int64)
    predicted = torch.tensor([], dtype=torch.int64)
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct   += (pred == y).sum().item()
            total     += y.size(0)
            # Accumulate all labels on CPU to build the confusion matrix after the loop
            actual    = torch.cat((actual,    y.cpu()),    dim=0)
            predicted = torch.cat((predicted, pred.cpu()), dim=0)

    acc = correct / total
    if plot:
        print(np.unique(predicted.numpy(), return_counts=True))
        cm = confusion_matrix(actual.numpy(), predicted.numpy())
        ConfusionMatrixDisplay(cm).plot()
    print(f'Test accuracy: {acc * 100:.2f}%')
    return acc

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# -- Paths --
DATA_DIR    = '/Volumes/KRIS/data/UG_per_subject'
MODEL_TYPE  = 'EEGNet'   # change to: 'EEGNet' or 'ShallowConvNet'
WEIGHTS_DIR = os.path.join(_notebook_dir, 'weights', MODEL_TYPE)
LOG_PATH    = os.path.join(_notebook_dir, f'results_log_{MODEL_TYPE}.txt')

# -- Data split --
BATCH_SIZE = 16
VAL_FRAC   = 0.10     # fraction of training pool held out for early stopping (stratified)

# -- Training --
MAX_EPOCHS = 20
PATIENCE   = 5
MIN_DELTA  = 0.002
LR         = 1e-3
DROPOUT    = 0.1

# os.makedirs(WEIGHTS_DIR, exist_ok=True)
# subjects = load_all_subjects(DATA_DIR)
# print(f'\nWeights → {WEIGHTS_DIR}')
# print(f'Log     → {LOG_PATH}')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

MODEL_MAP = {
    # 'EMG_TCN':        dlm.EMG_TCN,
    # 'EEGNet':         dlm.EEGNet,
    # 'ShallowConvNet': dlm.ShallowConvNet,
}

def subject_id(filename):
    """'emg_gestures_03_U.mat' → '03'"""
    return filename.replace('emg_gestures_', '').replace('_U.mat', '')


def weight_path(subject_name):
    return os.path.join(WEIGHTS_DIR, f'{MODEL_TYPE}_{subject_id(subject_name)}.pt')


def update_log():
    """Rebuild results_log_<MODEL_TYPE>.txt from all valid LOSO checkpoint files in WEIGHTS_DIR."""
    checkpoints = []
    for fname in sorted(os.listdir(WEIGHTS_DIR)):
        if not fname.endswith('.pt'):
            continue
        ckpt = torch.load(os.path.join(WEIGHTS_DIR, fname), map_location='cpu', weights_only=False)
        if 'subject' not in ckpt:
            continue
        checkpoints.append(ckpt)

    if not checkpoints:
        return

    accs    = [c['test_acc'] * 100 for c in checkpoints]
    n_done  = len(checkpoints)
    n_total = len(subjects)

    lines = [
        f'putEMG — {MODEL_TYPE} LOSO Results',
        '=' * 68,
        f"{'Subject':<28} {'Test Acc':>9}  {'Val Acc':>9}  {'Epoch':>6}  {'Date'}",
        '-' * 68,
    ]
    for c in checkpoints:
        lines.append(
            f"{c['subject']:<28} {c['test_acc']*100:>8.2f}%  "
            f"{c['val_acc']*100:>8.2f}%  {c['best_epoch']:>6}  {c['date']}"
        )
    lines += [
        '=' * 68,
        f"Mean: {np.mean(accs):.2f}%  ±  {np.std(accs):.2f}%  "
        f"({n_done} / {n_total} folds complete)",
    ]

    with open(LOG_PATH, 'w') as f:
        f.write('\n'.join(lines) + '\n')

    print(f"Log updated → {LOG_PATH}  ({n_done}/{n_total} folds)")